# ЛР3 — класифікація фракцій відходів на плитках БПЛА
Варіант 9: ViT-B/16, корекція дисбалансу — **фокальна функція втрат**.

**Дані:** у цій версії набір **синтетичний** — плитки 224×224, згенеровані скриптом
`synth_textures.py` для 8 «сцен» з різним освітленням, кольоровим зсувом, розмиттям і шумом.
Класи: `mineral, wood, metal, mixed, background`. Для реальних плиток достатньо підготувати
`cls_manifest.csv` (`path, scene, label`) — комірка генерації його не перезапише.

Ваги ViT-B/16: ImageNet (timm `vit_base_patch16_224`, файл `jx_vit_base_p16_224-80ecf9dd.pth`;
якщо файлу немає — завантажуються через `timm` автоматично).

In [1]:
import os, json, random, time
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms as T, models
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

SMOKE = os.environ.get("SMOKE") == "1"   # швидка перевірка коду без GPU/ваг
PROTOCOL = dict(seed=0, img_size=224, epochs_baseline=20, epochs_probe=12, epochs_finetune=8,
                batch=8, optimizer="AdamW", lr_head=1e-3, lr_body=1e-5, weight_decay=0.05,
                patience=4, val_metric="macro-F1", focal_gamma=2.0, norm_mean_std=0.5)
CLASSES = ["mineral", "wood", "metal", "mixed", "background"]
K = len(CLASSES)
DEV = "cuda" if torch.cuda.is_available() else "cpu"

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(PROTOCOL["seed"])
print(DEV, json.dumps(PROTOCOL, ensure_ascii=False))
json.dump(PROTOCOL, open("protocol.json", "w"), indent=1)

cpu {"seed": 0, "img_size": 224, "epochs_baseline": 20, "epochs_probe": 12, "epochs_finetune": 8, "batch": 8, "optimizer": "AdamW", "lr_head": 0.001, "lr_body": 1e-05, "weight_decay": 0.05, "patience": 4, "val_metric": "macro-F1", "focal_gamma": 2.0, "norm_mean_std": 0.5}


## 0. Генерація синтетичного набору

In [2]:
import sys; sys.path.append("../common"); sys.path.append(".")
if not os.path.exists("cls_manifest.csv"):
    from synth_textures import make_tile, scene_style
    rng = np.random.default_rng(7); rows = []
    os.makedirs("tiles", exist_ok=True)
    CLASS_P = [.28, .2, .1, .2, .22]          # природний дисбаланс: metal рідкісний
    for sc in range(8):
        st = scene_style(rng)
        for i in range(45):
            lab = CLASSES[rng.choice(K, p=CLASS_P)]
            pth = f"tiles/sc{sc:02d}_{i:03d}_{lab}.png"
            Image.fromarray(make_tile(lab, rng, st)).save(pth)
            rows.append(dict(path=pth, scene=f"sc{sc:02d}", label=lab))
    pd.DataFrame(rows).to_csv("cls_manifest.csv", index=False)
m0 = pd.read_csv("cls_manifest.csv"); print(len(m0), "плиток;", m0.scene.nunique(), "сцен"); print(m0.label.value_counts())
fig, ax = plt.subplots(2, 5, figsize=(14, 6))
for j, c in enumerate(CLASSES):
    for i in range(2):
        r = m0[m0.label == c].iloc[i * 7]; ax[i, j].imshow(Image.open(r.path)); ax[i, j].axis("off")
        ax[i, j].set_title(f"{c} ({r.scene})", fontsize=9)
plt.tight_layout(); plt.savefig("fig_examples.png", dpi=120); plt.show()

360 плиток; 8 сцен
label
mineral       103
background     87
wood           71
mixed          63
metal          36
Name: count, dtype: int64


## 1. Маніфест і групове розбиття за сценами (якщо `split` ще немає)

In [3]:
from sklearn.model_selection import GroupShuffleSplit
df = pd.read_csv("cls_manifest.csv")
if "split" not in df:
    for seed in range(200):   # шукаємо розбиття, де кожен клас є в кожній підмножині
        i_tr, i_rest = next(GroupShuffleSplit(1, test_size=0.3, random_state=seed).split(df, groups=df.scene))
        rest = df.iloc[i_rest]
        i_va, i_te = next(GroupShuffleSplit(1, test_size=0.5, random_state=seed).split(rest, groups=rest.scene))
        parts = [df.iloc[i_tr], rest.iloc[i_va], rest.iloc[i_te]]
        if all(set(p.label) == set(df.label) for p in parts): break
    df["split"] = "train"; df.loc[parts[1].index, "split"] = "val"; df.loc[parts[2].index, "split"] = "test"
    df.to_csv("cls_split.csv", index=False)
for s in ["val", "test"]:   # перевірка витоку
    print(s, "частка сцен, присутніх у train:",
          df[df.split == s].scene.isin(set(df[df.split == "train"].scene)).mean())
pd.crosstab(df.label, df.split)

val частка сцен, присутніх у train: 0.0
test частка сцен, присутніх у train: 0.0


split,test,train,val
label,,,
background,21,49,17
metal,8,25,3
mineral,28,66,9
mixed,14,42,7
wood,19,43,9


## 2. Набір даних і аугментації (вид згори: повороти на 90°, віддзеркалення, яскравість)

In [4]:
MEAN, STD = [0.5] * 3, [0.5] * 3          # нормалізація ваг ViT-B/16 (timm, jx)
S = PROTOCOL["img_size"]
class Rot90:
    def __call__(self, x): return x.rotate(90 * random.randint(0, 3))
train_tf = T.Compose([T.RandomResizedCrop(S, scale=(0.7, 1.0)), T.RandomHorizontalFlip(),
                      T.RandomVerticalFlip(), Rot90(), T.ColorJitter(0.3, 0.3),
                      T.ToTensor(), T.Normalize(MEAN, STD)])
eval_tf = T.Compose([T.Resize((S, S)), T.ToTensor(), T.Normalize(MEAN, STD)])

class TileDS(Dataset):
    def __init__(self, frame, tf):
        self.p = frame.path.tolist(); self.y = [CLASSES.index(l) for l in frame.label]; self.tf = tf
    def __len__(self): return len(self.p)
    def __getitem__(self, i): return self.tf(Image.open(self.p[i]).convert("RGB")), self.y[i]

def loaders(train_frame, sampler=None):
    g = torch.Generator().manual_seed(PROTOCOL["seed"])
    tr = DataLoader(TileDS(train_frame, train_tf), batch_size=PROTOCOL["batch"], shuffle=sampler is None,
                    sampler=sampler, num_workers=0, generator=g)
    va = DataLoader(TileDS(df[df.split == "val"], eval_tf), batch_size=64, num_workers=0)
    te = DataLoader(TileDS(df[df.split == "test"], eval_tf), batch_size=64, num_workers=0)
    return tr, va, te
train_df = df[df.split == "train"]
train_loader, val_loader, test_loader = loaders(train_df)
print({s: len(df[df.split == s]) for s in ["train", "val", "test"]})

{'train': 225, 'val': 45, 'test': 90}


## 3. Моделі: baseline (мала CNN з нуля) і ViT-B/16 (лінійний зонд / донавчання)

In [5]:
class SmallCNN(nn.Module):           # baseline, навчається з нуля
    def __init__(self, k=K):
        super().__init__()
        def blk(i, o): return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(), nn.MaxPool2d(2))
        self.f = nn.Sequential(blk(3, 32), blk(32, 64), blk(64, 128), blk(128, 128), nn.AdaptiveAvgPool2d(1))
        self.fc = nn.Sequential(nn.Flatten(), nn.Dropout(0.3), nn.Linear(128, k))
    def forward(self, x): return self.fc(self.f(x))

import timm
from timm.models.vision_transformer import checkpoint_filter_fn
VIT_WEIGHTS = "jx_vit_base_p16_224-80ecf9dd.pth"
def vit(mode):
    if os.path.exists(VIT_WEIGHTS):
        m = timm.create_model("vit_base_patch16_224", pretrained=False)
        m.load_state_dict(checkpoint_filter_fn(torch.load(VIT_WEIGHTS, map_location="cpu"), m))
    else:
        m = timm.create_model("vit_base_patch16_224", pretrained=not SMOKE)
    m.reset_classifier(K)                                          # нова голова на K класів
    head = list(m.head.parameters())
    body = [p for n, p in m.named_parameters() if not n.startswith("head.")]
    if mode == "probe":                                            # екстракція ознак
        for p in body: p.requires_grad = False
        opt = torch.optim.AdamW(head, lr=PROTOCOL["lr_head"])
    else:                                                          # донавчання всіх шарів, два темпи
        opt = torch.optim.AdamW([{"params": body, "lr": PROTOCOL["lr_body"]},
                                 {"params": head, "lr": PROTOCOL["lr_head"]}],
                                weight_decay=PROTOCOL["weight_decay"])
    return m, opt

class FocalLoss(nn.Module):          # (3.9)
    def __init__(self, gamma=2.0): super().__init__(); self.g = gamma
    def forward(self, logits, y):
        logp = F.log_softmax(logits, 1).gather(1, y[:, None])[:, 0]
        return (-(1 - logp.exp()) ** self.g * logp).mean()

@torch.no_grad()
def predict(model, loader):
    model.eval(); Y, P, PR = [], [], []
    for x, y in loader:
        pr = F.softmax(model(x.to(DEV)), 1).cpu()
        Y += y.tolist(); P += pr.argmax(1).tolist(); PR.append(pr)
    return np.array(Y), np.array(P), torch.cat(PR).numpy()

def train(name, model, opt, loss_fn, tr_loader, epochs):
    model.to(DEV); sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    hist = []; best, bad = -1, 0
    for ep in range(epochs):
        model.train(); tl, tc, tn = 0, 0, 0
        for x, y in tr_loader:
            x, y = x.to(DEV), y.to(DEV); opt.zero_grad()
            out = model(x); loss = loss_fn(out, y); loss.backward(); opt.step()
            tl += loss.item() * len(y); tc += (out.argmax(1) == y).sum().item(); tn += len(y)
        sched.step()
        model.eval(); vl, vn = 0, 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEV), y.to(DEV); vl += loss_fn(model(x), y).item() * len(y); vn += len(y)
        yv, pv, _ = predict(model, val_loader)
        vf1 = f1_score(yv, pv, average="macro"); vacc = accuracy_score(yv, pv)
        hist.append(dict(epoch=ep + 1, train_loss=tl / tn, train_acc=tc / tn, val_loss=vl / vn, val_acc=vacc, val_f1=vf1))
        print(f"[{name}] ep{ep+1:02d} loss {tl/tn:.3f}/{vl/vn:.3f} acc {tc/tn:.3f}/{vacc:.3f} valF1 {vf1:.3f}")
        if vf1 > best: best, bad = vf1, 0; torch.save(model.state_dict(), f"best_{name}.pt")
        else:
            bad += 1
            if bad >= PROTOCOL["patience"]: print("рання зупинка"); break
    model.load_state_dict(torch.load(f"best_{name}.pt", map_location=DEV))
    h = pd.DataFrame(hist); h.to_csv(f"hist_{name}.csv", index=False)
    return model, h

/usr/local/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 4. Навчання: baseline → ViT probe → ViT fine-tune

In [6]:
ce = nn.CrossEntropyLoss()
results, H = {}, {}
set_seed(0); m = SmallCNN(); opt = torch.optim.AdamW(m.parameters(), lr=1e-3, weight_decay=1e-4)
models_ = {}
models_["baseline"], H["baseline"] = train("baseline", m, opt, ce, train_loader, PROTOCOL["epochs_baseline"])
for mode in ["probe", "finetune"]:
    set_seed(0); m, opt = vit(mode)
    models_[f"vit_{mode}"], H[f"vit_{mode}"] = train(f"vit_{mode}", m, opt, ce, train_loader, PROTOCOL[f"epochs_{mode}"])

[baseline] ep01 loss 1.565/1.284 acc 0.289/0.556 valF1 0.276


[baseline] ep02 loss 1.482/1.022 acc 0.351/0.800 valF1 0.716


[baseline] ep03 loss 1.340/0.895 acc 0.409/0.600 valF1 0.390


[baseline] ep04 loss 1.363/0.923 acc 0.400/0.600 valF1 0.459


[baseline] ep05 loss 1.385/0.812 acc 0.418/0.667 valF1 0.484


[baseline] ep06 loss 1.267/0.857 acc 0.480/0.600 valF1 0.476
рання зупинка


... (15 outputs left out)

[vit_finetune] ep05 loss 0.036/0.114 acc 0.996/0.978 valF1 0.975


[vit_finetune] ep06 loss 0.043/0.138 acc 0.996/0.956 valF1 0.931


[vit_finetune] ep07 loss 0.021/0.118 acc 1.000/0.978 valF1 0.975


[vit_finetune] ep08 loss 0.023/0.118 acc 1.000/0.978 valF1 0.975


In [7]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for n, h in H.items():
    l, = ax[0].plot(h.epoch, h.train_loss, "-", label=f"{n} train")
    ax[0].plot(h.epoch, h.val_loss, "--", color=l.get_color(), label=f"{n} val")
    ax[1].plot(h.epoch, h.val_acc, label=n)
ax[0].set(xlabel="епоха", ylabel="функція втрат", title="суцільні — train, штрихові — val"); ax[0].legend(fontsize=7)
ax[1].set(xlabel="епоха", ylabel="accuracy (val)"); ax[1].legend()
plt.tight_layout(); plt.savefig("fig_curves.png", dpi=150); plt.show()
val_table = pd.DataFrame({n: dict(best_val_f1=h.val_f1.max(), best_val_acc=h.val_acc.max(), epochs=len(h))
                          for n, h in H.items()}).T
display(val_table); val_table.to_csv("val_table.csv")

,best_val_f1,best_val_acc,epochs
baseline,0.715606,0.800000,6.0
vit_probe,0.959664,0.977778,11.0
vit_finetune,0.974902,0.977778,8.0


## 5. Тест — ОДИН раз, найкраща за валідацією модель

In [8]:
def wilson(p, n, z=1.96):
    c = z * z / n; centre = (p + c / 2) / (1 + c)
    half = z * np.sqrt(p * (1 - p) / n + c / (4 * n)) / (1 + c); return centre - half, centre + half

best_name = val_table.best_val_f1.idxmax(); print("найкраща модель:", best_name)
yt, pt, prt = predict(models_[best_name], test_loader)
acc = accuracy_score(yt, pt); lo, hi = wilson(acc, len(yt))
print(f"Accuracy = {acc:.3f}, 95% ДІ Вілсона = [{lo:.3f}; {hi:.3f}], n = {len(yt)}")
print(classification_report(yt, pt, labels=range(K), target_names=CLASSES, digits=3, zero_division=0))
cm = confusion_matrix(yt, pt, labels=range(K))
plt.figure(figsize=(5, 4.5)); plt.imshow(cm, cmap="Greys")
for i in range(K):
    for j in range(K): plt.text(j, i, cm[i, j], ha="center", va="center", color="red")
plt.xticks(range(K), CLASSES, rotation=45); plt.yticks(range(K), CLASSES)
plt.xlabel("передбачений"); plt.ylabel("істинний"); plt.tight_layout(); plt.savefig("fig_cm.png", dpi=150); plt.show()
off = cm.copy(); np.fill_diagonal(off, 0); i, j = np.unravel_index(off.argmax(), off.shape)
print("найчастіша помилка:", CLASSES[i], "->", CLASSES[j], off[i, j])

найкраща модель: vit_finetune


Accuracy = 0.989, 95% ДІ Вілсона = [0.940; 0.998], n = 90
              precision    recall  f1-score   support

     mineral      1.000     1.000     1.000        28
        wood      1.000     1.000     1.000        19
       metal      0.889     1.000     0.941         8
       mixed      1.000     0.929     0.963        14
  background      1.000     1.000     1.000        21

    accuracy                          0.989        90
   macro avg      0.978     0.986     0.981        90
weighted avg      0.990     0.989     0.989        90

найчастіша помилка: mixed -> metal 1


## 6. Дисбаланс: штучно зменшуємо клас `metal` до 20 % у train;
порівнюємо звичайну CE і фокальну функцію втрат (γ = 2). Модель — ViT probe (швидко).

In [9]:
RARE = "metal"; KEEP = 0.2
rare_idx = train_df[train_df.label == RARE].sample(frac=KEEP, random_state=0).index
imb_df = pd.concat([train_df[train_df.label != RARE], train_df.loc[rare_idx]])
print(imb_df.label.value_counts())
imb_loader, _, _ = loaders(imb_df)
imb = {}
for lname, lfn in [("CE", nn.CrossEntropyLoss()), ("focal", FocalLoss(PROTOCOL["focal_gamma"]))]:
    set_seed(0); m, opt = vit("probe")
    m, _ = train(f"imb_{lname}", m, opt, lfn, imb_loader, PROTOCOL["epochs_probe"])
    y, p, _ = predict(m, test_loader)
    rec = (p[y == CLASSES.index(RARE)] == CLASSES.index(RARE)).mean() if (y == CLASSES.index(RARE)).any() else np.nan
    imb[lname] = dict(macro_F1=f1_score(y, p, average="macro"), recall_rare=rec, accuracy=accuracy_score(y, p))
imb = pd.DataFrame(imb).T; display(imb); imb.to_csv("imbalance.csv")

label
mineral       66
background    49
wood          43
mixed         42
metal          5
Name: count, dtype: int64


[imb_CE] ep01 loss 0.775/0.553 acc 0.712/0.889 valF1 0.722


[imb_CE] ep02 loss 0.291/0.374 acc 0.937/0.889 valF1 0.722


[imb_CE] ep03 loss 0.207/0.322 acc 0.946/0.889 valF1 0.722


[imb_CE] ep04 loss 0.178/0.321 acc 0.961/0.867 valF1 0.698


[imb_CE] ep05 loss 0.148/0.275 acc 0.951/0.911 valF1 0.744


... (13 outputs left out)

[imb_focal] ep10 loss 0.041/0.093 acc 0.961/0.933 valF1 0.853


[imb_focal] ep11 loss 0.021/0.093 acc 0.995/0.911 valF1 0.744


[imb_focal] ep12 loss 0.022/0.094 acc 0.980/0.911 valF1 0.744
рання зупинка


,macro_F1,recall_rare,accuracy
CE,0.756491,0.000,0.911111
focal,0.797302,0.125,0.911111


## 7. Аналіз помилок: найвпевненіші і найменш впевнені помилки + Grad-CAM

In [10]:
test_df = df[df.split == "test"].reset_index(drop=True)
err = np.where(yt != pt)[0]
conf = prt[err, pt[err]]
order = err[np.argsort(-conf)]
show = list(order[:8]) + list(order[-8:])
fig, ax = plt.subplots(2, 8, figsize=(18, 5))
for a, k in zip(ax.ravel(), show):
    a.imshow(Image.open(test_df.path[k])); a.axis("off")
    a.set_title(f"{CLASSES[yt[k]]}→{CLASSES[pt[k]]}\n{prt[k, pt[k]]:.2f} {test_df.scene[k]}", fontsize=7)
plt.suptitle("верхній ряд — найвпевненіші помилки (можливий шум розмітки), нижній — найменш впевнені")
plt.tight_layout(); plt.savefig("fig_errors.png", dpi=150); plt.show()
print("помилки за сценами (перевірка на доменний зсув):")
display(test_df.loc[err].scene.value_counts())

помилки за сценами (перевірка на доменний зсув):


scene
sc06    1
Name: count, dtype: int64

In [11]:
if not SMOKE:
    !pip -q install grad-cam
    from pytorch_grad_cam import GradCAM
    from pytorch_grad_cam.utils.image import show_cam_on_image
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
    def reshape_vit(t, h=14, w=14):     # токени ViT -> карта 14x14
        r = t[:, 1:, :].reshape(t.size(0), h, w, t.size(2)); return r.permute(0, 3, 1, 2)
    m = models_[best_name]
    if best_name.startswith("vit"):
        for p in m.parameters(): p.requires_grad = True   # для Grad-CAM потрібні градієнти
        cam = GradCAM(model=m, target_layers=[m.blocks[-1].norm1], reshape_transform=reshape_vit)
    else:
        cam = GradCAM(model=m, target_layers=[m.f[3]])
    fig, ax = plt.subplots(1, 6, figsize=(18, 3.5))
    for a, k in zip(ax, list(order[:6])):
        img = Image.open(test_df.path[k]).convert("RGB").resize((S, S))
        g = cam(input_tensor=eval_tf(img)[None].to(DEV), targets=[ClassifierOutputTarget(int(pt[k]))])[0]
        a.imshow(show_cam_on_image(np.asarray(img) / 255., g, use_rgb=True)); a.axis("off")
        a.set_title(f"{CLASSES[yt[k]]}→{CLASSES[pt[k]]}", fontsize=8)
    plt.tight_layout(); plt.savefig("fig_gradcam.png", dpi=150); plt.show()

## 8. (Опційно) розкид за зернами
Перезапусти розділ 4 для найкращої моделі з `seed = 1, 2` і впиши у звіт середнє ± std.